# Shendure MWU power — all cell types

Assesses MWU power (TPR) as a function of CRE_oi strength relative to `minP`, for every cell type in the Shendure dataset.

For each cell type:
1. Draw `n_cres` synthetic CREs from uniform(min, max), with one fixed at `minP` as the reference
2. Simulate `cells_per_cell_type` cells (matching the real count for that cell type) across `n_sims` replicates, repeated 100 times
3. Run MWU for all CREs vs. reference
4. Aggregate and plot per-cell-type power curves

Each cell type is simulated independently with its own empirical `(n_cres, cell_count)` pair.
`one_library_replicate` now accepts a `cell_type` parameter so real names flow through cleanly.

In [ ]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [ ]:
data_root = Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
sim_date = "2026-03-13"

In [ ]:
local = False
if local:
    cluster = LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster = SLURMCluster(
        cores=3,
        memory="16G",
        processes=1,
        job_extra_directives=[
            "-p day",
            "--job-name=simclust_worker",
            "--time=12:00:00",
            "--output=worker_%j.out"
        ]
    )
    cluster.scale(jobs=10)
    client = Client(
        cluster,
        timeout=f"{5*60}s",
        heartbeat_interval="20s"
    )
print(client.dashboard_link, flush=True)

In [ ]:
# Load primordial to get per-cell-type n_cres from post-ortho_filter training data.
# For Shendure (integrating vector + bottleneck), detected n_cres per cell type
# reflects CREs that genuinely survived into that lineage.
primordial = scm.ortho.load(client, data_root / "shendure", "ortho_primordial_v4")
dat = primordial.training_data.data.compute()

cell_types = sorted(dat["cell_type"].unique().tolist())
n_cres_per_ct = {
    ct: len(dat[dat["cell_type"] == ct]["cre_id"].unique())
    for ct in cell_types
}
del primordial

print("Cell types and parameters:")
for ct in cell_types:
    cells = scm.SHENDURE_BOUNDS.cells_per_cell_type.get(ct, "N/A")
    print(f"  {ct}: n_cres={n_cres_per_ct[ct]}, cells={cells}")

In [ ]:
max_activity = 0.05
min_activity = scm.SHENDURE_BOUNDS.min_mpra_umi
minP = scm.SHENDURE_BOUNDS.reference_activity
n_library_reps = 100
n_sims = 5
print(f"minP={minP:.6f}, min={min_activity:.6f}, max={max_activity}")

In [ ]:
all_sims = {}  # {cell_type: [sim, ...]}

for cell_type in cell_types:
    print(f"\n=== {cell_type} ===", flush=True)

    bound_ct = scm.SHENDURE_BOUNDS.copy()
    bound_ct.cells_per_cell_type = scm.SHENDURE_BOUNDS.cells_per_cell_type.loc[[cell_type]]

    sims_ct = []
    for i in range(n_library_reps):
        _, sim = scm.one_library_replicate(
            root=data_root / f"{sim_date}_shendure_pow" / cell_type,
            n_sims=n_sims,
            client=client,
            flatten_overtransfection=False,
            bound=bound_ct,
            n_cres=n_cres_per_ct[cell_type],
            min=min_activity,
            max=max_activity,
            minP=minP,
            cell_type=cell_type,
        )
        sims_ct.append(sim)

    all_sims[cell_type] = sims_ct
    print(f"  {n_library_reps} replicates done", flush=True)

In [ ]:
for sims in all_sims.values():
    for sim in sims:
        sim.save()
print("Saved.", flush=True)

In [ ]:
# Build a hypothesis set per cell type from its first simulation's data.
for cell_type, sims in all_sims.items():
    print(f"MWU: {cell_type}", flush=True)
    example = scm.scMPRA_data.from_parquet(sims[0].scmpradatp / "0.scmpra")
    hs = scm.make_all_by_celltype_hypotheses(counts=example, reference_cre="reference")
    for sim in sims:
        sim.add_hypothesis_set("hs_all_ct", hs)
        sim.mwu("hs_all_ct")

for sims in all_sims.values():
    for sim in sims:
        sim.save()
print("MWU done and saved.", flush=True)

In [ ]:
all_mergy = {
    ct: scm.sum_pow(sims, hypothesis_set_name="hs_all_ct", test_type="mwu")
    for ct, sims in all_sims.items()
}

In [ ]:
def _pow_curve_data(mergy, n_bins=100):
    df = mergy.copy()
    df["fc"] = pd.cut(df["fc"], bins=n_bins)
    binned = (
        df.groupby("fc", observed=True)["reject_null"]
        .mean()
        .reset_index(name="reject_frac")
    )
    binned["bin_center"] = binned["fc"].apply(lambda x: x.mid)
    return binned

ncols = 3
nrows = math.ceil(len(cell_types) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharey=True)
axes = axes.flatten()
palette = sns.color_palette("tab20", n_colors=len(cell_types))

for i, cell_type in enumerate(cell_types):
    binned = _pow_curve_data(all_mergy[cell_type])
    ax = axes[i]
    ax.plot(binned["bin_center"], binned["reject_frac"],
            color=palette[i], marker="o", markersize=2, linewidth=1)
    ax.axhline(0.8, color="black", linestyle="--", lw=0.8)
    ax.axvline(1.0, color="grey", linestyle=":", lw=0.8)
    ax.set_title(cell_type, fontsize=9)
    ax.set_xlabel("FC (activity / minP)", fontsize=8)
    ax.set_ylabel("Power (TPR)", fontsize=8)
    ax.set_ylim(0, 1)
    cells = scm.SHENDURE_BOUNDS.cells_per_cell_type.get(cell_type, "?")
    ax.text(0.97, 0.05,
            f"n_cres={n_cres_per_ct[cell_type]}, cells={cells}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=6, color="grey")

for j in range(len(cell_types), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("MWU Power by Cell Type — Shendure", fontsize=12)
plt.tight_layout()
svg_path = output_dir / "power_mwu_all_cell_types.svg"
fig.savefig(svg_path, format="svg", bbox_inches="tight")
print(f"Saved: {svg_path}")
plt.show()

In [ ]:
client.close()
cluster.close()